<a href="https://colab.research.google.com/github/MZiaAfzal71/Average_Weighted_Path_Vector/blob/main/Data%20Files/Models/ChemBERTaFiLMFusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/MZiaAfzal71/Average_Weighted_Path_Vector.git
%cd Average_Weighted_Path_Vector/Data\ Files
!pip install scikit-learn rdkit osfclient

In [ ]:
from osfclient.api import OSF
import os
from subprocess import run

# Replace with your OSF project ID
project_id = "p5ga2"   # e.g. from https://osf.io/abcd3/
osf = OSF()
project = osf.project(project_id)
store = project.storage("osfstorage")

desc_folder = []
for fold in store.folders:
    if fold.path.strip("/") == "Descriptors Data":
        desc_folder = fold
        break

# Download all files and keep folder structure
for f in desc_folder.files:
    local_path = f.path.strip("/")            # keep folders
    local_dir = os.path.dirname(local_path)   # extract dir
    if local_dir and not os.path.exists(local_dir):
        os.makedirs(local_dir, exist_ok=True) # create dirs if missing
    with open(local_path, "wb") as out:
        f.write_to(out)
    if local_path.endswith(".zip"):
      command = f"unzip '{local_path}' -d '{local_dir}'"
      run(command, shell=True)
      print(f"\nUnzipped {local_path} -> {local_dir}")
      continue
    print(f"Downloaded {f.path} -> {local_path}")

In [ ]:
# ============================
# ChemBERTa + Descriptor Gated Training Pipeline
# ============================
from __future__ import annotations
import random, math, json, gc
from dataclasses import dataclass
from typing import List, Optional, Dict, Any, Tuple
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

import csv
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tqdm.auto import tqdm

In [ ]:
def safe_name(x: str) -> str:
    return str(x).replace(" ", "").replace("/", "_").replace("\\", "_")

def murcko_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        return None

def murcko_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        return None

def get_split_indices(df: pd.DataFrame, split_mode="benchmark", test_size=0.2, random_state=42):
    if split_mode == "benchmark":
        train_idx = df.index[df["Training/Test"].astype(str).str.strip().str.lower() == "training"].tolist()
        test_idx = df.index[df["Training/Test"].astype(str).str.strip().str.lower() == "test"].tolist()

        split_info = df[["SMILES"]].copy()
        split_info["split"] = df["Training/Test"] if "Training/Test" in df.columns else "unknown"
        split_info["split_mode"] = "benchmark"
        return train_idx, test_idx, split_info

    elif split_mode == "scaffold":
        scaffold_df = df[["SMILES"]].copy()
        scaffold_df["scaffold"] = scaffold_df["SMILES"].apply(murcko_scaffold)

        valid_mask = scaffold_df["scaffold"].notna()
        valid_idx = scaffold_df.index[valid_mask]
        groups = scaffold_df.loc[valid_idx, "scaffold"]

        if len(valid_idx) == 0:
            raise ValueError("No valid Murcko scaffolds generated.")

        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_pos, test_pos = next(gss.split(valid_idx, groups=groups))

        train_idx = valid_idx[train_pos].tolist()
        test_idx = valid_idx[test_pos].tolist()

        split_info = scaffold_df.copy()
        split_info["split"] = "unused"
        split_info.loc[train_idx, "split"] = "Training"
        split_info.loc[test_idx, "split"] = "Test"
        split_info["split_mode"] = "scaffold"
        return train_idx, test_idx, split_info

    else:
        raise ValueError(f"Unknown split_mode: {split_mode}")

def get_feature_columns(df: pd.DataFrame, prop: str, desc_name: str):
    target_col = f"{prop}-Measured"
    all_cols = list(df.columns)

    meta_cols = {
        "Preferred_name", "SMILES", "Training/Test", f"{prop}-EPISuite Prediction",
        f"{prop}-Prediction from our model", "CAS RN",	"NAME",	"IUPAC Name",
        target_col
    }

    if desc_name in {"PWAV_raw", "PWAV_pruned"}:
        pwav_cols = [c for c in all_cols if c.startswith(f"{prop}_PWAV_")]
        extra_cols = [c for c in all_cols if c.startswith(f"{prop}_EXTRA_")]
        return {
            "target_col": target_col,
            "pwav_cols": pwav_cols,
            "extra_cols": extra_cols,
            "direct_feature_cols": None
        }
    else:
        direct_feature_cols = [c for c in all_cols if c not in meta_cols]
        return {
            "target_col": target_col,
            "pwav_cols": None,
            "extra_cols": None,
            "direct_feature_cols": direct_feature_cols
        }


def prepare_descriptor_arrays(df, prop, desc_name, train_idx, test_idx, cfg):
    col_info = get_feature_columns(df, prop, desc_name)

    train_df = df.loc[train_idx].reset_index(drop=True).copy()
    test_df = df.loc[test_idx].reset_index(drop=True).copy()

    pca_model = None

    if desc_name in {"PWAV_raw", "PWAV_pruned"}:
        pwav_cols = col_info["pwav_cols"]
        extra_cols = col_info["extra_cols"]

        X_train_core = train_df[pwav_cols].to_numpy(dtype=np.float32)
        X_test_core = test_df[pwav_cols].to_numpy(dtype=np.float32)

        if cfg.n_pca is not None and cfg.n_pca < X_train_core.shape[1]:
            pca_model = PCA(n_components=cfg.n_pca, random_state=cfg.seed)
            X_train_core = pca_model.fit_transform(X_train_core)
            X_test_core = pca_model.transform(X_test_core)

        if extra_cols:
            X_train_extra = train_df[extra_cols].to_numpy(dtype=np.float32)
            X_test_extra = test_df[extra_cols].to_numpy(dtype=np.float32)
            X_train = np.hstack([X_train_core, X_train_extra]).astype(np.float32)
            X_test = np.hstack([X_test_core, X_test_extra]).astype(np.float32)
        else:
            X_train = X_train_core.astype(np.float32)
            X_test = X_test_core.astype(np.float32)

    else:
        direct_cols = col_info["direct_feature_cols"]
        X_train = train_df[direct_cols].to_numpy(dtype=np.float32)
        X_test = test_df[direct_cols].to_numpy(dtype=np.float32)

    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    return train_df, test_df, X_train, X_test, scaler, pca_model, col_info

In [ ]:
# ----------------------------
# Config
# ----------------------------
@dataclass
class Config:
    model_name: str = "seyonec/ChemBERTa-zinc-base-v1"
    log_path: str = "training_log.csv"
    output_dir: str = "./chemberta_fusion_output"
    save_path: str = "best_model.pt"

    max_length: int = 128
    batch_size: int = 16
    epochs: int = 30
    lr_backbone: float = 1e-5
    lr_heads: float = 1e-4
    weight_decay: float = 0.01
    seed: int = 42
    hidden_fuse: int = 512
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    proj_dim: int = 128
    dropout: float = 0.1

    split_mode: str = "benchmark"   # "benchmark" or "scaffold"
    test_size: float = 0.2
    descriptor_name: str = "PWAV_raw"

    train_mode: str = "last2"       # "full", "last2", "head_only"
    fusion_mode: str = "full"       # "full", "concat", "film_only", "gating_only"
    stress_test: str = "none"       # "none", "shuffle_desc", "drop_desc", "drop_smiles", "freeze_gate"

    gate_temp: float = 1.0
    p_moddrop: float = 0.2
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0
    lambda_aux: float = 0.2
    lambda_div: float = 0.05
    n_pca: int = 64
    return_numpy: bool = True
# ----------------------------
# Utils
# ----------------------------
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def configure_backbone_trainability(backbone, train_mode="full"):
    for p in backbone.parameters():
        p.requires_grad = False

    if train_mode == "head_only":
        return

    if train_mode == "full":
        for p in backbone.parameters():
            p.requires_grad = True
        return

    if hasattr(backbone, "encoder") and hasattr(backbone.encoder, "layer"):
        if train_mode == "last2":
            for layer in backbone.encoder.layer[-2:]:
                for p in layer.parameters():
                    p.requires_grad = True
            return

    # fallback
    if train_mode != "head_only":
        for p in backbone.parameters():
            p.requires_grad = True

# ----------------------------
# Model (gated fusion)
# ----------------------------

class ChemBERTaFusion(nn.Module):
    """
    ChemBERTa + descriptors with configurable fusion modes:

    fusion_mode:
      - "full"        : modality dropout + FiLM + gating + aux heads + diversity penalty
      - "concat"      : concatenate CLS and descriptor projection, then regress
      - "film_only"   : FiLM-conditioned CLS only
      - "gating_only" : gate raw CLS and descriptor projection without FiLM

    stress_test:
      - "none"
      - "drop_desc"   : zero out descriptor branch
      - "drop_smiles" : zero out ChemBERTa branch
      - "freeze_gate" : gate fixed at 0.5
    """
    def __init__(
        self,
        model_name: str,
        n_desc: int,
        proj_dim: int = 128,
        hidden_fuse: int = 512,
        dropout: float = 0.1,
        train_mode: str = "full",
        fusion_mode: str = "full",
        stress_test: str = "none",
        gate_temp: float = 1.0,
        p_moddrop: float = 0.2,
    ):
        super().__init__()

        self.backbone = AutoModel.from_pretrained(model_name)
        H = self.backbone.config.hidden_size
        self.n_desc = n_desc
        self.gate_temp = gate_temp
        self.p_moddrop = p_moddrop
        self.fusion_mode = fusion_mode
        self.stress_test = stress_test
        self.hidden_size = H

        # trainability
        configure_backbone_trainability(self.backbone, train_mode=train_mode)

        # Descriptor projection into backbone hidden space
        self.desc_proj = nn.Sequential(
            nn.Linear(n_desc, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, H),
            nn.LayerNorm(H),
            nn.ReLU(),
        )

        # Gating MLP
        self.gate_mlp = nn.Sequential(
            nn.Linear(2 * H, hidden_fuse),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_fuse, H),
        )

        # FiLM conditioning: desc -> (gamma, beta)
        self.film = nn.Linear(H, 2 * H)

        self.dropout = nn.Dropout(dropout)

        # Main head input depends on fusion mode
        main_in = 2 * H if fusion_mode == "concat" else H
        self.head_main = nn.Sequential(
            nn.Linear(main_in, H // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(H // 2, 1),
        )

        # Auxiliary heads
        self.head_cls = nn.Linear(H, 1)
        self.head_desc = nn.Linear(H, 1)

    def forward(
        self,
        input_ids,
        attention_mask,
        descriptors,
        targets=None,
        lambda_aux=0.2,
        lambda_div=0.05,
    ):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]         # [B, H]
        desc_h = self.desc_proj(descriptors)         # [B, H]

        B = cls.size(0)

        # Optional train-time modality dropout (only for full mode)
        if self.training and self.fusion_mode == "full" and self.p_moddrop > 0.0:
            mask_choice = torch.empty(B, 1, device=cls.device).uniform_(0, 1)
            drop_cls = (mask_choice < self.p_moddrop).float()
            drop_desc = (mask_choice > 1.0 - self.p_moddrop).float()
            cls = cls * (1.0 - drop_cls)
            desc_h = desc_h * (1.0 - drop_desc)

        # Stress tests
        if self.stress_test == "drop_smiles":
            cls = torch.zeros_like(cls)
        elif self.stress_test == "drop_desc":
            desc_h = torch.zeros_like(desc_h)

        # FiLM
        gamma_beta = self.film(desc_h)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        cls_mod = (1.0 + gamma) * cls + beta

        g = None
        cos_ref_cls = cls_mod if self.fusion_mode == "full" else cls

        # Fusion selection
        if self.fusion_mode == "concat":
            fused = torch.cat([cls, desc_h], dim=-1)

        elif self.fusion_mode == "film_only":
            fused = cls_mod

        elif self.fusion_mode == "gating_only":
            gate_logits = self.gate_mlp(torch.cat([cls, desc_h], dim=-1)) / self.gate_temp
            g = torch.sigmoid(gate_logits)
            if self.stress_test == "freeze_gate":
                g = torch.full_like(g, 0.5)
            fused = g * cls + (1.0 - g) * desc_h

        else:  # "full"
            gate_logits = self.gate_mlp(torch.cat([cls_mod, desc_h], dim=-1)) / self.gate_temp
            g = torch.sigmoid(gate_logits)
            if self.stress_test == "freeze_gate":
                g = torch.full_like(g, 0.5)
            fused = g * cls_mod + (1.0 - g) * desc_h

        fused = self.dropout(fused)

        y_main = self.head_main(fused).squeeze(-1)
        y_cls = self.head_cls(cls_mod).squeeze(-1)
        y_desc = self.head_desc(desc_h).squeeze(-1)

        loss = None
        diag = {}

        if targets is not None:
            targets = targets.float()

            l_main = F.mse_loss(y_main, targets)
            l_cls = F.mse_loss(y_cls, targets)
            l_desc = F.mse_loss(y_desc, targets)

            eps = 1e-8
            c_sim = F.cosine_similarity(
                F.normalize(cos_ref_cls, dim=-1, eps=eps),
                F.normalize(desc_h, dim=-1, eps=eps),
                dim=-1,
            ).mean()

            # Keep loss logic consistent across modes
            if self.fusion_mode in {"concat", "film_only", "gating_only", "full"}:
                loss = l_main + lambda_aux * (l_cls + l_desc) + lambda_div * c_sim
            else:
                loss = l_main

            diag = {
                "loss_main": float(l_main.detach().item()),
                "loss_cls": float(l_cls.detach().item()),
                "loss_desc": float(l_desc.detach().item()),
                "cos_sim": float(c_sim.detach().item()),
                "gate_mean": float(g.mean().detach().item()) if g is not None else np.nan,
            }

        return y_main, loss, diag



# ----------------------------
# Dataset / Collate
# ----------------------------
class SmiDescDataset(Dataset):
    def __init__(self, smiles: List[str], targets: Optional[np.ndarray],
                 tokenizer: AutoTokenizer, max_length: int,
                 descriptors: Optional[np.ndarray] = None):
        self.smiles = list(smiles)
        self.targets = None if targets is None else np.asarray(targets, dtype=np.float32)
        self.tok = tokenizer
        self.max_length = max_length
        self.desc = None if descriptors is None else np.asarray(descriptors, dtype=np.float32)

    def __len__(self): return len(self.smiles)

    def __getitem__(self, i):
        enc = self.tok(self.smiles[i],
                       truncation=True, padding="max_length",
                       max_length=self.max_length, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.targets is not None:
            item["labels"] = torch.tensor(self.targets[i], dtype=torch.float32)
        if self.desc is not None:
            # ensure finite floats
            d = self.desc[i]
            if not np.all(np.isfinite(d)):  # replace bad values
                d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)
            item["descriptors"] = torch.tensor(d, dtype=torch.float32)
        return item

def collate_stack(batch):
    out = {k: torch.stack([b[k] for b in batch]) for k in batch[0] if k != "labels"}
    if "labels" in batch[0]:
        out["labels"] = torch.stack([b["labels"] for b in batch])
    return out


def make_fusion_loaders(df, prop, desc_name, tokenizer, cfg):
    target_col = f"{prop}-Measured"

    train_idx, test_idx, split_info = get_split_indices(
        df,
        split_mode=cfg.split_mode,
        test_size=cfg.test_size,
        random_state=cfg.seed
    )

    train_df, test_df, X_train_desc, X_test_desc, scaler, pca_model, col_info = prepare_descriptor_arrays(
        df, prop, desc_name, train_idx, test_idx, cfg
    )

    train_ds = SmiDescDataset(
        train_df["SMILES"].tolist(),
        train_df[target_col].to_numpy(dtype=np.float32),
        tokenizer,
        cfg.max_length,
        X_train_desc
    )
    test_ds = SmiDescDataset(
        test_df["SMILES"].tolist(),
        test_df[target_col].to_numpy(dtype=np.float32),
        tokenizer,
        cfg.max_length,
        X_test_desc
    )

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_stack)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_stack)

    return train_loader, test_loader, train_df, test_df, split_info, scaler, pca_model, col_info


def setup_optimizer_scheduler(model, train_dataloader, epochs, lr=2e-5, lr_head=1e-3, warmup_ratio=0.1):

    # separate ChemBERTa backbone vs fusion head params
    backbone_params, head_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "backbone" in n:
            backbone_params.append(p)
        else:
            head_params.append(p)
    optimizer = torch.optim.AdamW([ {"params": backbone_params, "lr": lr},
                                   {"params": head_params, "lr": lr_head}, ],
                                  weight_decay=0.01)

    total_steps = len(train_dataloader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    return optimizer, scheduler

def train_model(model, train_loader, val_loader, optimizer, scheduler,
                device, epochs=10, grad_clip=1.0, lambda_aux=0.2, lambda_div=0.05,
                save_path="best_model.pt", log_path="training_log.csv"):
    """
    Full trainer loop for ChemBERTaFusion with richer CSV logging.
    """
    model.to(device)
    best_val = float("inf")

    # --- Prepare CSV log ---
    log_file = Path(log_path)
    write_header = not log_file.exists()

    with open(log_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow([
                "epoch", "train_loss", "val_loss",
                "train_loss_cls", "train_loss_desc",
                "val_loss_cls", "val_loss_desc",
                "cos_sim", "gate_mean", "val_mae"
            ])

    for epoch in range(1, epochs+1):
        # ---- TRAIN ----
        model.train()
        train_loss, n_train = 0.0, 0
        train_loss_cls, train_loss_desc = 0.0, 0.0
        diag_accum = {"cos_sim": 0.0, "gate_mean": 0.0}

        pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")
        for batch in pbar:
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            descriptors = batch["descriptors"].to(device)
            targets = batch["labels"].to(device)

            preds, loss, diag = model(
                input_ids, attention_mask, descriptors,
                targets, lambda_aux=lambda_aux, lambda_div=lambda_div
            )

            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()

            bs = input_ids.size(0)
            train_loss += loss.item() * bs
            n_train += bs

            train_loss_cls += diag["loss_cls"] * bs
            train_loss_desc += diag["loss_desc"] * bs
            diag_accum["cos_sim"] += diag["cos_sim"] * bs
            diag_accum["gate_mean"] += diag["gate_mean"] * bs

            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss /= n_train
        train_loss_cls /= n_train
        train_loss_desc /= n_train
        diag_accum = {k: v / n_train for k, v in diag_accum.items()}

        # ---- VALIDATION ----
        model.eval()
        val_loss, n_val = 0.0, 0
        val_loss_cls, val_loss_desc, val_mae = 0.0, 0.0, 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                descriptors = batch["descriptors"].to(device)
                targets = batch["labels"].to(device)

                preds, loss, diag = model(
                    input_ids, attention_mask, descriptors,
                    targets, lambda_aux=lambda_aux, lambda_div=lambda_div
                )

                bs = input_ids.size(0)
                val_loss += loss.item() * bs
                val_loss_cls += diag["loss_cls"] * bs
                val_loss_desc += diag["loss_desc"] * bs
                # MAE on predictions
                val_mae += F.l1_loss(preds, targets, reduction="sum").item()
                n_val += bs

        val_loss /= n_val
        val_loss_cls /= n_val
        val_loss_desc /= n_val
        val_mae /= n_val

        print(
            f"Epoch {epoch}: "
            f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
            f"train_cls={train_loss_cls:.4f}, val_cls={val_loss_cls:.4f}, "
            f"train_desc={train_loss_desc:.4f}, val_desc={val_loss_desc:.4f}, "
            f"cos_sim={diag_accum['cos_sim']:.3f}, gate_mean={diag_accum['gate_mean']:.3f}, "
            f"val_mae={val_mae:.4f}"
        )

        # ---- Log to CSV ----
        with open(log_file, mode="a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch,
                train_loss, val_loss,
                train_loss_cls, train_loss_desc,
                val_loss_cls, val_loss_desc,
                diag_accum["cos_sim"], diag_accum["gate_mean"],
                val_mae
            ])

        # ---- Save best ----
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved best model (val_loss={val_loss:.4f})")

    print("Training complete.")
    model.load_state_dict(torch.load(save_path))
    return model

def predict(model, test_loader, device, return_numpy=True):
    """
    Run inference on a test set.

    Args:
      model: trained ChemBERTaFusionV2
      test_loader: DataLoader
      device: torch.device
      return_numpy: if True, returns numpy array

    Returns:
      preds: [N] predictions (torch.Tensor or np.ndarray)
    """
    model.eval()
    model.to(device)
    all_preds = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            descriptors = batch["descriptors"].to(device)

            preds, _, _ = model(input_ids, attention_mask, descriptors)
            all_preds.append(preds.cpu())

    preds = torch.cat(all_preds, dim=0)

    if return_numpy:
        return preds.numpy()
    return preds


def train_gated_for_prop_desc(prop: str, desc_name: str, cfg: Config) -> Dict[str, Any]:
    set_seed(cfg.seed)
    ensure_dir(cfg.output_dir)

    safe_prop = safe_name(prop)
    run_name = (
        f"{safe_prop}_{desc_name}_{cfg.split_mode}_"
        f"{cfg.train_mode}_{cfg.fusion_mode}_{cfg.stress_test}"
    )

    target_col = f"{prop}-Measured"
    data_file = f"Descriptors Data/{safe_prop}_{desc_name}.parquet"

    if not os.path.exists(data_file):
        raise ValueError(f"{data_file} is not found.")

    df = pd.read_parquet(data_file)

    # tokenizer
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)

    # split
    train_idx, test_idx, split_info = get_split_indices(
        df,
        split_mode=cfg.split_mode,
        test_size=cfg.test_size,
        random_state=cfg.seed,
    )

    # descriptor prep: train-only PCA + train-only scaling
    train_df, test_df, X_train_desc, X_test_desc, scaler, pca_model, col_info = prepare_descriptor_arrays(
        df, prop, desc_name, train_idx, test_idx, cfg
    )

    # optional descriptor shuffling stress test on TEST set only
    if cfg.stress_test == "shuffle_desc":
        rng = np.random.default_rng(cfg.seed)
        perm = rng.permutation(len(X_test_desc))
        X_test_desc = X_test_desc[perm]

    # datasets / loaders
    train_ds = SmiDescDataset(
        train_df["SMILES"].tolist(),
        train_df[target_col].to_numpy(dtype=np.float32),
        tokenizer,
        cfg.max_length,
        X_train_desc,
    )
    test_ds = SmiDescDataset(
        test_df["SMILES"].tolist(),
        test_df[target_col].to_numpy(dtype=np.float32),
        tokenizer,
        cfg.max_length,
        X_test_desc,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_stack,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_stack,
    )

    # model
    n_desc = X_train_desc.shape[1]
    model = ChemBERTaFusion(
        model_name=cfg.model_name,
        n_desc=n_desc,
        proj_dim=cfg.proj_dim,
        hidden_fuse=cfg.hidden_fuse,
        dropout=cfg.dropout,
        train_mode=cfg.train_mode,
        fusion_mode=cfg.fusion_mode,
        stress_test=cfg.stress_test,
        gate_temp=cfg.gate_temp,
        p_moddrop=cfg.p_moddrop,
    ).to(cfg.device)

    optimizer, scheduler = setup_optimizer_scheduler(
        model,
        train_loader,
        cfg.epochs,
        cfg.lr_backbone,
        cfg.lr_heads,
        cfg.warmup_ratio,
    )

    save_model = os.path.join(cfg.output_dir, f"{run_name}_{cfg.save_path}")
    save_log = os.path.join(cfg.output_dir, f"{run_name}_{cfg.log_path}")

    model = train_model(
        model,
        train_loader,
        test_loader,
        optimizer,
        scheduler,
        cfg.device,
        cfg.epochs,
        cfg.grad_clip,
        cfg.lambda_aux,
        cfg.lambda_div,
        save_model,
        save_log,
    )

    # final test predictions
    final_test_preds = predict(model, test_loader, cfg.device, cfg.return_numpy)
    y_test = test_df[target_col].to_numpy(dtype=np.float32)

    mae_v = mean_absolute_error(y_test, final_test_preds)
    rmse_v = rmse(y_test, final_test_preds)
    r2_v = r2_score(y_test, final_test_preds)

    print(
        f"Final Test metrics for {run_name} → "
        f"MAE: {mae_v:.4f} | RMSE: {rmse_v:.4f} | R²: {r2_v:.4f}"
    )

    # save split metadata
    split_path = os.path.join(cfg.output_dir, f"{run_name}_split.csv")
    split_info.to_csv(split_path, index=False)

    # save PCA metadata if used
    if pca_model is not None:
        pca_meta = {
            "property": prop,
            "descriptor": desc_name,
            "split_mode": cfg.split_mode,
            "train_mode": cfg.train_mode,
            "fusion_mode": cfg.fusion_mode,
            "stress_test": cfg.stress_test,
            "n_components": int(pca_model.n_components_),
            "explained_variance_ratio_sum": float(np.sum(pca_model.explained_variance_ratio_)),
        }
        pca_path = os.path.join(cfg.output_dir, f"{run_name}_pca.json")
        with open(pca_path, "w") as f:
            json.dump(pca_meta, f, indent=2)

    # save test predictions
    test_results = pd.DataFrame({
        "Preferred_name": test_df["Preferred_name"] if "Preferred_name" in test_df.columns else pd.Series([None] * len(test_df)),
        "SMILES": test_df["SMILES"],
        "Observed": y_test,
        "Predicted": final_test_preds,
        "split_mode": cfg.split_mode,
        "train_mode": cfg.train_mode,
        "fusion_mode": cfg.fusion_mode,
        "stress_test": cfg.stress_test,
    })
    test_pred_path = os.path.join(cfg.output_dir, f"{run_name}_test_predictions.parquet")
    test_results.to_parquet(test_pred_path, index=False)

    # all-row predictions using train-fitted preprocessing
    if desc_name in {"PWAV_raw", "PWAV_pruned"}:
        pwav_cols = col_info["pwav_cols"]
        extra_cols = col_info["extra_cols"]

        X_all_core = df[pwav_cols].to_numpy(dtype=np.float32)
        if pca_model is not None:
            X_all_core = pca_model.transform(X_all_core)

        if extra_cols:
            X_all_extra = df[extra_cols].to_numpy(dtype=np.float32)
            X_all_desc = np.hstack([X_all_core, X_all_extra]).astype(np.float32)
        else:
            X_all_desc = X_all_core.astype(np.float32)
    else:
        X_all_desc = df[col_info["direct_feature_cols"]].to_numpy(dtype=np.float32)

    X_all_desc = scaler.transform(X_all_desc).astype(np.float32)

    # important: do NOT apply shuffle_desc to all-row predictions
    all_ds = SmiDescDataset(
        df["SMILES"].tolist(),
        df[target_col].to_numpy(dtype=np.float32),
        tokenizer,
        cfg.max_length,
        X_all_desc,
    )
    all_loader = DataLoader(
        all_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_stack,
    )

    all_preds = predict(model, all_loader, cfg.device, cfg.return_numpy)

    all_results = pd.DataFrame({
        "Preferred_name": df["Preferred_name"] if "Preferred_name" in df.columns else pd.Series([None] * len(df)),
        "SMILES": df["SMILES"],
        "Observed": df[target_col],
        "Predicted": all_preds,
        "Training/Test": df["Training/Test"] if "Training/Test" in df.columns else pd.Series([None] * len(df)),
        "split_mode": cfg.split_mode,
        "train_mode": cfg.train_mode,
        "fusion_mode": cfg.fusion_mode,
        "stress_test": cfg.stress_test,
    })
    all_pred_path = os.path.join(cfg.output_dir, f"{run_name}_all_predictions.parquet")
    all_results.to_parquet(all_pred_path, index=False)

    return {
        "property": prop,
        "descriptor": desc_name,
        "split_mode": cfg.split_mode,
        "train_mode": cfg.train_mode,
        "fusion_mode": cfg.fusion_mode,
        "stress_test": cfg.stress_test,
        "MAE": mae_v,
        "RMSE": rmse_v,
        "R2": r2_v,
        "n_train": len(train_df),
        "n_test": len(test_df),
        "descriptor_dim": int(n_desc),
        "best_path": save_model,
        "test_pred_path": test_pred_path,
        "all_pred_path": all_pred_path,
    }

In [ ]:
cfg = Config(
    output_dir="chemberta_fusion_results",
    epochs=30,
    batch_size=8,
    max_length=128,
    train_mode="last2",
    split_mode="benchmark",
    descriptor_name="PWAV_raw",
    fusion_mode="gating_only",               # "concat", "film_only", "gating_only", "full"
    stress_test="none",
    lr_backbone=1e-5,
    lr_heads=1e-4,
    n_pca=64,
    seed=42,
)

prop_names = ["BP", "LogS", "MP"]
desc_names = ["PWAV_raw"]

perf_rows = []
for prop in prop_names:
    for desc_name in desc_names:
        res = train_gated_for_prop_desc(prop, desc_name, cfg)
        perf_rows.append(res)

perf_df = pd.DataFrame(perf_rows)
perf_df.to_csv(os.path.join(cfg.output_dir, "fusion_debug_run.csv"), index=False)
perf_df